# Modul 05: Bild- und Signaldaten vorbereiten | Lösungen

## Überblick

Sie arbeiten mit synthetischen Bild- und Signaldaten, untersuchen Datenformen, Farbräume, Pixelwerte, Zeitachsen und Abtastraten und erzeugen anschließend kompakte Merkmale. Die Aufgaben verbinden Pillow, OpenCV, NumPy, pandas und Matplotlib in kleinen, Colab-freundlichen Workflows.

**Zugehörige Vorlesungen**

- **Bilddaten vorbereiten**
- **Signale vorbereiten**

## Lernziele

Nach der Bearbeitung können Sie:

- Bilder mit Pillow, OpenCV und NumPy laden beziehungsweise konvertieren und Farbräume, Kanäle und Wertebereiche prüfen.
- Zuschneiden, Skalieren, Filtern, Kanten bestimmen und eine einheitliche Bildvorverarbeitung implementieren.
- Zeitreihen mit Abtastrate erzeugen, Fenster- und Änderungsmerkmale berechnen und dominante Frequenzen bestimmen.

## Geprüfte Fähigkeiten

- RGB/BGR, Graustufen, Alpha, Normalisierung, Histogramme, Resize und Kanten
- Zeitachse, Abtastrate, Rauschen, gleitende Fenster und lokale Kennzahlen
- FFT-basierte Frequenzmerkmale und tabellarische Merkmalsausgabe

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** mittel
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Bilder und Signale werden synthetisch erzeugt. Dadurch sind Formate und Störungen kontrollierbar, und es ist kein Daten-Download notwendig.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Synthetisches RGB-Bild mit Farbflächen und einer hellen Diagonale.
bild_rgb = np.zeros((96, 128, 3), dtype=np.uint8)
bild_rgb[:, :43] = [220, 60, 60]
bild_rgb[:, 43:86] = [60, 190, 90]
bild_rgb[:, 86:] = [60, 90, 220]
for i in range(min(bild_rgb.shape[:2])):
    bild_rgb[i, i] = [255, 255, 255]

# Ein Signal mit zwei Frequenzanteilen, Trend und Rauschen.
abtastrate_hz = 100
zeit_s = np.arange(0, 4, 1 / abtastrate_hz)
signal_sauber = 1.2 * np.sin(2 * np.pi * 5 * zeit_s) + 0.45 * np.sin(2 * np.pi * 12 * zeit_s)
trend = 0.08 * zeit_s
signal = signal_sauber + trend + rng.normal(0, 0.18, size=zeit_s.size)

print("Einrichtung abgeschlossen.")
print("RGB-Bildform:", bild_rgb.shape, "Datentyp:", bild_rgb.dtype)
print("Signalpunkte:", len(signal), "Abtastrate:", abtastrate_hz, "Hz")

### Aufgabe 1: Bildbibliotheken und Farbräume sicher verwenden

1. Erzeugen Sie aus `bild_rgb` ein Pillow-Bild.
2. Simulieren Sie die OpenCV-Darstellung durch Umwandlung von RGB nach BGR und wieder zurück.
3. Prüfen Sie Form, Datentyp und drei ausgewählte Pixelwerte vor und nach der Rückumwandlung.
4. Zeigen Sie Original und korrekt zurückgewandeltes Bild nebeneinander.
5. Prüfen Sie mit `np.array_equal`, ob die Rückumwandlung verlustfrei war.

In [ ]:
# bild_rgb liegt als NumPy-Array in RGB-Reihenfolge vor.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Pillow erwartet RGB in derselben Kanalreihenfolge wie das Ausgangsarray.
pil_bild = Image.fromarray(bild_rgb, mode="RGB")

# OpenCV verwendet für viele Farbbildoperationen standardmäßig BGR.
bild_bgr = cv2.cvtColor(bild_rgb, cv2.COLOR_RGB2BGR)
bild_rgb_zurueck = cv2.cvtColor(bild_bgr, cv2.COLOR_BGR2RGB)

print("Pillow-Modus:", pil_bild.mode)
print("Pillow-Größe (Breite, Höhe):", pil_bild.size)
print("NumPy-Form:", bild_rgb.shape)
print("BGR-Pixel links oben:", bild_bgr[0, 0])
print("RGB-Pixel zurück:", bild_rgb_zurueck[0, 0])
print("Verlustfreie Rückumwandlung:", np.array_equal(bild_rgb, bild_rgb_zurueck))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(bild_rgb)
axes[0].set_title("Original in RGB")
axes[0].axis("off")

axes[1].imshow(bild_rgb_zurueck)
axes[1].set_title("BGR zurück nach RGB")
axes[1].axis("off")
plt.tight_layout()
plt.show()

assert np.array_equal(bild_rgb, bild_rgb_zurueck)

> **Musterantwort und Interpretation**
>
> Rot- und Blaukanal werden vertauscht. Das Bild kann formal korrekt aussehen, zeigt aber falsche Farben. Deshalb sollten Kanalreihenfolge und Bibliotheksannahmen früh technisch und visuell geprüft werden.

### Aufgabe 2: Graustufen, Normalisierung, Histogramm und Zuschnitt

1. Wandeln Sie das Bild in Graustufen um.
2. Normalisieren Sie die Pixelwerte auf den Bereich 0 bis 1 und verwenden Sie `float32`.
3. Berechnen Sie ein Histogramm mit 16 Klassen für die Graustufenwerte.
4. Schneiden Sie den mittleren Bildbereich aus und skalieren Sie ihn auf 32 × 32 Pixel.
5. Geben Sie alle relevanten Formen und Wertebereiche aus und visualisieren Sie Graubild, Zuschnitt und Histogramm.

In [ ]:
# Nutzen Sie OpenCV oder eine nachvollziehbare NumPy-Formel für die Graustufenumwandlung.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# cvtColor berücksichtigt die RGB-Kanalreihenfolge korrekt.
grau_uint8 = cv2.cvtColor(bild_rgb, cv2.COLOR_RGB2GRAY)

# Durch die explizite Typumwandlung vermeiden wir ganzzahlige Divisionseffekte.
grau_norm = grau_uint8.astype(np.float32) / 255.0

# Histogramm über den ursprünglichen 0-bis-255-Bereich.
hist_werte, hist_kanten = np.histogram(grau_uint8, bins=16, range=(0, 256))

# Der zentrale Bereich wird über Zeilen- und Spaltenslices gewählt.
hoehe, breite = grau_uint8.shape
zuschnitt = grau_uint8[hoehe // 4 : 3 * hoehe // 4, breite // 4 : 3 * breite // 4]
zuschnitt_32 = cv2.resize(zuschnitt, (32, 32), interpolation=cv2.INTER_AREA)

print("Graubildform:", grau_uint8.shape)
print("Graubildtyp:", grau_uint8.dtype)
print("Normalisierter Bereich:", float(grau_norm.min()), "bis", float(grau_norm.max()))
print("Zuschnittform:", zuschnitt.shape)
print("Skalierte Form:", zuschnitt_32.shape)
print("Histogrammsumme:", int(hist_werte.sum()), "Pixel")

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(grau_uint8, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Graustufenbild")
axes[0].axis("off")

axes[1].imshow(zuschnitt_32, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Mittlerer Zuschnitt, 32 × 32")
axes[1].axis("off")

# Die Balkenbreite folgt den Histogrammkanten.
axes[2].bar(hist_kanten[:-1], hist_werte, width=np.diff(hist_kanten), align="edge")
axes[2].set_title("Grauwert-Histogramm")
axes[2].set_xlabel("Grauwert")
axes[2].set_ylabel("Pixelanzahl")
plt.tight_layout()
plt.show()

### Aufgabe 3: Filter, Kanten und wiederverwendbare Bildvorverarbeitung

Implementieren Sie `preprocess_bild(rgb_array, zielgroesse=(48, 48))`. Die Funktion soll:

- Eingabeform `(H, W, 3)` und Datentyp prüfen,
- nach Graustufen umwandeln,
- mit einem kleinen Gauß-Filter glätten,
- Canny-Kanten berechnen,
- Graubild und Kanten auf `zielgroesse` skalieren,
- das Graubild auf 0 bis 1 normalisieren,
- ein Dictionary mit `grau`, `kanten`, `mittlere_helligkeit` und `kantendichte` zurückgeben.

Visualisieren Sie das Ergebnis und prüfen Sie Formen sowie Wertebereiche.

In [ ]:
def preprocess_bild(rgb_array, zielgroesse=(48, 48)):
    """Bereitet ein RGB-Bild reproduzierbar für eine kleine Analyse vor."""
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def preprocess_bild(rgb_array, zielgroesse=(48, 48)):
    """Bereitet ein RGB-Bild reproduzierbar für eine kleine Analyse vor."""
    if not isinstance(rgb_array, np.ndarray):
        raise TypeError("rgb_array muss ein NumPy-Array sein.")
    if rgb_array.ndim != 3 or rgb_array.shape[2] != 3:
        raise ValueError("Erwartet wird die Form (Höhe, Breite, 3).")
    if rgb_array.dtype != np.uint8:
        raise ValueError("Für dieses Beispiel wird uint8 im Bereich 0 bis 255 erwartet.")

    # Graustufen reduzieren drei Farbkanäle auf eine Helligkeitsachse.
    grau = cv2.cvtColor(rgb_array, cv2.COLOR_RGB2GRAY)

    # Glättung reduziert einzelne Pixelstörungen vor der Kantenerkennung.
    geglaettet = cv2.GaussianBlur(grau, (5, 5), sigmaX=0)
    kanten = cv2.Canny(geglaettet, threshold1=50, threshold2=140)

    # cv2.resize erwartet die Zielgröße als (Breite, Höhe).
    grau_klein = cv2.resize(grau, zielgroesse, interpolation=cv2.INTER_AREA)
    kanten_klein = cv2.resize(kanten, zielgroesse, interpolation=cv2.INTER_NEAREST)

    grau_norm = grau_klein.astype(np.float32) / 255.0
    kantendichte = float(np.mean(kanten_klein > 0))

    return {
        "grau": grau_norm,
        "kanten": kanten_klein,
        "mittlere_helligkeit": float(grau_norm.mean()),
        "kantendichte": kantendichte,
    }


ergebnis_bild = preprocess_bild(bild_rgb)
print("Grausform:", ergebnis_bild["grau"].shape)
print("Kantenform:", ergebnis_bild["kanten"].shape)
print("Graubereich:", ergebnis_bild["grau"].min(), "bis", ergebnis_bild["grau"].max())
print("Mittlere Helligkeit:", round(ergebnis_bild["mittlere_helligkeit"], 3))
print("Kantendichte:", round(ergebnis_bild["kantendichte"], 3))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(ergebnis_bild["grau"], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Vorverarbeitetes Graubild")
axes[0].axis("off")
axes[1].imshow(ergebnis_bild["kanten"], cmap="gray")
axes[1].set_title("Canny-Kanten")
axes[1].axis("off")
plt.tight_layout()
plt.show()

assert ergebnis_bild["grau"].shape == (48, 48)
assert 0.0 <= ergebnis_bild["kantendichte"] <= 1.0

> **Musterantwort und Interpretation**
>
> Nur eine einheitliche Vorverarbeitung macht Merkmale zwischen Bildern vergleichbar. Werden Schwellen oder Größen unkontrolliert pro Bild verändert, können Unterschiede aus dem Verfahren statt aus dem Bildinhalt entstehen.

### Aufgabe 4: Zeitachse, Abtastrate und verrauschtes Signal

1. Prüfen Sie rechnerisch, ob die Zeitachse zur Abtastrate passt.
2. Visualisieren Sie die ersten zwei Sekunden des Signals.
3. Berechnen Sie Dauer, Mittelwert, Standardabweichung, Minimum und Maximum.
4. Erzeugen Sie zusätzlich ein Signal mit gleicher Dauer, aber nur 25 Hz Abtastrate. Verwenden Sie dieselben Frequenzkomponenten ohne Rauschen.
5. Vergleichen Sie die Anzahl der Messpunkte und erläutern Sie, welche Signalanteile bei niedrigerer Abtastrate schwieriger darstellbar werden.

In [ ]:
# zeit_s, signal und abtastrate_hz wurden in der Einrichtungszelle erstellt.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Die Zeitabstände müssen im Mittel 1 / Abtastrate betragen.
zeitabstand = np.diff(zeit_s)
print("Mittlerer Zeitabstand:", zeitabstand.mean())
print("Erwarteter Zeitabstand:", 1 / abtastrate_hz)
assert np.allclose(zeitabstand, 1 / abtastrate_hz)

maske_zwei_sekunden = zeit_s < 2.0
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(zeit_s[maske_zwei_sekunden], signal[maske_zwei_sekunden])
ax.set_title("Verrauschtes Signal, erste zwei Sekunden")
ax.set_xlabel("Zeit in Sekunden")
ax.set_ylabel("Amplitude")
plt.show()

kennzahlen = pd.Series(
    {
        "Dauer_s": zeit_s[-1] - zeit_s[0] + 1 / abtastrate_hz,
        "Mittelwert": signal.mean(),
        "Standardabweichung": signal.std(),
        "Minimum": signal.min(),
        "Maximum": signal.max(),
    }
)
print(kennzahlen.round(3))

# Eine zweite Zeitachse verwendet deutlich weniger Messpunkte pro Sekunde.
abtastrate_niedrig = 25
zeit_niedrig = np.arange(0, 4, 1 / abtastrate_niedrig)
signal_niedrig = (
    1.2 * np.sin(2 * np.pi * 5 * zeit_niedrig)
    + 0.45 * np.sin(2 * np.pi * 12 * zeit_niedrig)
    + 0.08 * zeit_niedrig
)

print("Punkte bei 100 Hz:", len(zeit_s))
print("Punkte bei 25 Hz:", len(zeit_niedrig))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(zeit_niedrig, signal_niedrig, marker="o", markersize=2)
ax.set_title("Dasselbe Signalprinzip mit 25 Hz")
ax.set_xlabel("Zeit in Sekunden")
ax.set_ylabel("Amplitude")
plt.show()

> **Musterantwort und Interpretation**
>
> Nach dem Nyquist-Prinzip muss die Abtastrate mehr als doppelt so hoch wie die höchste relevante Frequenz sein. 25 Hz liegt nur knapp über 24 Hz, sodass der 12-Hz-Anteil mit sehr wenigen Punkten pro Schwingung dargestellt wird und praktisch sehr empfindlich gegenüber Rauschen oder kleinen Abweichungen ist.

### Aufgabe 5: Gleitende Fenster, lokale Merkmale und Qualitätsprobleme

1. Erzeugen Sie überlappende Fenster mit Länge 50 und Schrittweite 25.
2. Berechnen Sie je Fenster Startzeit, Mittelwert, Standardabweichung, Minimum, Maximum, RMS und mittlere absolute Änderung.
3. Fügen Sie in einer Kopie des Signals drei Fehlwerte und einen starken Ausreißer ein.
4. Ersetzen Sie Fehlwerte durch lineare Interpolation mit pandas.
5. Markieren Sie den Ausreißer mit einer robusten Median-Abweichungsregel oder einer klar erklärten Schwelle.
6. Erstellen Sie eine Merkmalstabelle.

In [ ]:
fensterlaenge = 50
schrittweite = 25
signal_problem = signal.copy()
signal_problem[[40, 41, 205]] = np.nan
signal_problem[260] = 6.5

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# pandas.interpolate() nutzt benachbarte Messwerte und erhält die Zeitreihenlänge.
signal_serie = pd.Series(signal_problem)
signal_bereinigt = signal_serie.interpolate(method="linear", limit_direction="both").to_numpy()

# Robuste Abweichung: Median und MAD werden kaum durch einen einzelnen Extremwert verzerrt.
median = np.median(signal_bereinigt)
mad = np.median(np.abs(signal_bereinigt - median))
robuster_z = 0.6745 * (signal_bereinigt - median) / mad
ist_ausreisser = np.abs(robuster_z) > 5.0

print("Ersetzte Fehlwerte:", int(np.isnan(signal_problem).sum()))
print("Markierte Ausreißerpunkte:", int(ist_ausreisser.sum()))

# Für die Merkmalsbildung ersetzen wir den markierten Extremwert vorsichtig durch den Median.
signal_fuer_merkmale = signal_bereinigt.copy()
signal_fuer_merkmale[ist_ausreisser] = median

zeilen = []
for start in range(0, len(signal_fuer_merkmale) - fensterlaenge + 1, schrittweite):
    ende = start + fensterlaenge
    fenster = signal_fuer_merkmale[start:ende]
    zeilen.append(
        {
            "start_index": start,
            "startzeit_s": zeit_s[start],
            "mittelwert": fenster.mean(),
            "standardabweichung": fenster.std(),
            "minimum": fenster.min(),
            "maximum": fenster.max(),
            "rms": np.sqrt(np.mean(fenster ** 2)),
            "mittlere_abs_aenderung": np.mean(np.abs(np.diff(fenster))),
        }
    )

fenster_merkmale = pd.DataFrame(zeilen)
display(fenster_merkmale.head())
print("Merkmalstabellenform:", fenster_merkmale.shape)

> **Musterantwort und Interpretation**
>
> Kurze Fenster reagieren schnell auf lokale Änderungen, liefern aber oft instabilere Kennzahlen und schlechtere Frequenzauflösung. Lange Fenster glätten lokale Unterschiede und sind stabiler. Eine kleine Schrittweite erzeugt mehr überlappende Beispiele und höheren Rechenaufwand; bei späteren Splits kann starke Überlappung außerdem zu Datenleckage führen.

### Aufgabe 6: Dominante Frequenzen und Signal-Merkmalstabelle

1. Entfernen Sie pro Fenster den Mittelwert, bevor Sie die FFT berechnen.
2. Bestimmen Sie für jedes Fenster die stärkste positive Frequenz ungleich 0 Hz.
3. Ergänzen Sie `dominante_frequenz_hz` und `dominante_amplitude` in der Merkmalstabelle.
4. Visualisieren Sie das Frequenzspektrum des ersten Fensters.
5. Prüfen Sie, ob häufig ungefähr 5 Hz oder 12 Hz erkannt werden.

In [ ]:
# Verwenden Sie signal_fuer_merkmale, fensterlaenge, schrittweite und fenster_merkmale aus Aufgabe 5.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

dominante_frequenzen = []
dominante_amplituden = []

for start in range(0, len(signal_fuer_merkmale) - fensterlaenge + 1, schrittweite):
    fenster = signal_fuer_merkmale[start : start + fensterlaenge]

    # Mittelwert entfernen, damit die Gleichkomponente bei 0 Hz nicht dominiert.
    fenster_zentriert = fenster - fenster.mean()
    spektrum = np.fft.rfft(fenster_zentriert)
    frequenzen = np.fft.rfftfreq(fensterlaenge, d=1 / abtastrate_hz)
    amplituden = np.abs(spektrum)

    # Index 0 gehört zu 0 Hz und wird bewusst ausgeschlossen.
    dominanter_index = 1 + np.argmax(amplituden[1:])
    dominante_frequenzen.append(frequenzen[dominanter_index])
    dominante_amplituden.append(amplituden[dominanter_index])

fenster_merkmale = fenster_merkmale.copy()
fenster_merkmale["dominante_frequenz_hz"] = dominante_frequenzen
fenster_merkmale["dominante_amplitude"] = dominante_amplituden

display(fenster_merkmale.head())
print("Häufigste dominante Frequenzen:")
print(fenster_merkmale["dominante_frequenz_hz"].value_counts().sort_index())

# Spektrum des ersten Fensters als kontrollierbares Beispiel.
erstes_fenster = signal_fuer_merkmale[:fensterlaenge]
erstes_zentriert = erstes_fenster - erstes_fenster.mean()
erstes_spektrum = np.abs(np.fft.rfft(erstes_zentriert))
erste_frequenzen = np.fft.rfftfreq(fensterlaenge, d=1 / abtastrate_hz)

fig, ax = plt.subplots(figsize=(9, 4))
ax.stem(erste_frequenzen, erstes_spektrum, basefmt=" ")
ax.set_xlim(0, 25)
ax.set_title("Frequenzspektrum des ersten Fensters")
ax.set_xlabel("Frequenz in Hz")
ax.set_ylabel("Amplitude")
plt.show()

> **Musterantwort und Interpretation**
>
> Ein längeres Fenster enthält einen längeren Beobachtungszeitraum. Dadurch liegen die diskreten FFT-Frequenzpunkte enger beieinander, und nahe Frequenzen können besser getrennt werden. Kurze Fenster liefern dagegen bessere zeitliche Lokalisierung, aber gröbere Frequenzauflösung.

### Aufgabe 7: Integrationsaufgabe: Bild- und Signalsnapshot dokumentieren

Erstellen Sie einen einzeiligen DataFrame `snapshot_merkmale`, der für den aktuellen Bild- und Signalsnapshot mindestens enthält:

- Bildhöhe und -breite,
- mittlere Bildhelligkeit,
- Kantendichte,
- Signaldauer,
- Signal-RMS,
- mittlere absolute Änderung,
- dominante Gesamtfrequenz.

Nutzen Sie Ihre Vorverarbeitungsfunktion für das Bild und eine FFT über das bereinigte Gesamtsignal. Ergänzen Sie eine kurze Qualitätsnotiz.

In [ ]:
# Nutzen Sie ergebnis_bild sowie das bereinigte Signal aus den vorherigen Aufgaben.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Bildmerkmale kommen aus genau derselben reproduzierbaren Funktion wie zuvor.
bild_features = preprocess_bild(bild_rgb, zielgroesse=(48, 48))

# Für Gesamtfrequenzen entfernen wir den Mittelwert und verwenden das vollständige bereinigte Signal.
gesamt_zentriert = signal_fuer_merkmale - signal_fuer_merkmale.mean()
gesamt_spektrum = np.abs(np.fft.rfft(gesamt_zentriert))
gesamt_frequenzen = np.fft.rfftfreq(len(gesamt_zentriert), d=1 / abtastrate_hz)
gesamt_index = 1 + np.argmax(gesamt_spektrum[1:])

dominante_gesamtfrequenz = float(gesamt_frequenzen[gesamt_index])

snapshot_merkmale = pd.DataFrame(
    [
        {
            "bildhoehe": bild_rgb.shape[0],
            "bildbreite": bild_rgb.shape[1],
            "mittlere_bildhelligkeit": bild_features["mittlere_helligkeit"],
            "kantendichte": bild_features["kantendichte"],
            "signaldauer_s": len(signal_fuer_merkmale) / abtastrate_hz,
            "signal_rms": np.sqrt(np.mean(signal_fuer_merkmale ** 2)),
            "mittlere_abs_aenderung": np.mean(np.abs(np.diff(signal_fuer_merkmale))),
            "dominante_frequenz_hz": dominante_gesamtfrequenz,
            "qualitaetsnotiz": "Drei Fehlwerte interpoliert; robuster Ausreißer vor Merkmalsbildung ersetzt.",
        }
    ]
)

display(snapshot_merkmale.round(4))

> **Musterantwort und Interpretation**
>
> Erforderlich sind mindestens Quelle, Aufnahmezeitpunkt, Geräte- oder Sensor-ID, Bildfarbraum, ursprüngliche Bildgröße, Abtastrate, Einheiten, verwendete Filterparameter, Fensterregeln und Softwareversionen. Außerdem muss dokumentiert werden, welche Fehlwerte oder Ausreißer wie behandelt wurden.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?